# Wildfires & The Spatial Partition Particle Filter

Welcome to the final interactive notebook of the `digital-twins` repository.

We introduce a severe limitation of the standard Particle Filter when applied to massive, spatiotemporal systems like a **wildfire**.

### The Curse of Spatial Locality (Figure 8.9)
Imagine a massive wildfire burning across hundreds of square miles. You have 1,000 particles trying to guess the shape of the fire. 
- Particle A guesses the Northern fire front perfectly, but is completely wrong about the Southern fire front.
- Particle B guesses the Southern fire front perfectly, but is completely wrong about the Northern fire front.

Because the standard Particle Filter calculates a **single, global importance weight**, *both* Particle A and Particle B will receive a mediocre "average" weight. During resampling, the algorithm might kill them both off! We lose perfectly good local information because of global averaging.

### The Solution: Spatial Partitioning
Instead of one global weight, we divide the map into a grid (e.g., 2x2). 
1. We calculate weights *locally* using only the sensors inside each grid square.
2. Particle A gets a massive weight in the North grid.
3. Particle B gets a massive weight in the South grid.
4. During resampling, we **stitch together** the North half of Particle A with the South half of Particle B, creating a new, highly accurate "Frankenstein" particle.

In this notebook, you will toggle between standard and spatial partitioning to see this stitching process in action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, Dropdown

### Interactive Learning: Stitching the Fire Front

**The Scenario:** A wildfire has two distinct active fronts (Northwest and Southeast). We have a grid of 40x40 cells and 16 temperature sensors. 

We generate a set of particles where no single particle has the whole picture right. 
- **Particle 1** is highly accurate in the Northwest, but fails in the Southeast.
- **Particle 2** is highly accurate in the Southeast, but fails in the Northwest.

**Instructions:**
1. Set Filter Mode to **'Standard Particle Filter (Global Weight)'**. Notice how the filter chooses the "least bad" particle, but it ultimately fails to capture the true fire accurately because it's a compromise.
2. Change Filter Mode to **'Spatial Partition PF (2x2 Grid)'**. Watch what happens! The filter draws a grid, scores the quadrants independently, and stitches the best quadrants together to perfectly recreate the true fire.

In [ ]:
def interactive_spatial_partition(filter_mode='Standard Particle Filter (Global Weight)'):
    # --- 1. Setup the Environment ---
    GRID_SIZE = 40
    cmap = ListedColormap(['#E0F8E0', '#FF4500']) # Light Green (Unburned), Orange-Red (Burning)
    
    # Create the TRUE Fire State (Two distinct fronts)
    y, x = np.ogrid[0:GRID_SIZE, 0:GRID_SIZE]
    true_fire = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    true_fire[((x - 10)**2 + (y - 10)**2 <= 6**2)] = 1  # NW Fire
    true_fire[((x - 30)**2 + (y - 30)**2 <= 7**2)] = 1  # SE Fire
    
    # --- 2. Create the Particles (Imperfect Guesses) ---
    # Particle 1: Gets NW right, completely wrong on SE
    p1 = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    p1[((x - 10)**2 + (y - 10)**2 <= 6**2)] = 1 
    p1[((x - 20)**2 + (y - 35)**2 <= 5**2)] = 1 # Wrong SE guess
    
    # Particle 2: Gets SE right, completely wrong on NW
    p2 = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    p2[((x - 25)**2 + (y - 5)**2 <= 5**2)] = 1  # Wrong NW guess
    p2[((x - 30)**2 + (y - 30)**2 <= 7**2)] = 1 
    
    # Particle 3: Mediocre everywhere (a typical compromised particle)
    p3 = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    p3[((x - 15)**2 + (y - 15)**2 <= 8**2)] = 1
    p3[((x - 25)**2 + (y - 25)**2 <= 8**2)] = 1
    
    particles = [p1, p2, p3]
    
    # --- 3. Deploy Sensors & Get Measurements ---
    # 16 sensors evenly deployed
    sensors_x, sensors_y = np.meshgrid(np.arange(5, 40, 10), np.arange(5, 40, 10))
    sensors_x, sensors_y = sensors_x.flatten(), sensors_y.flatten()
    
    # Very simplified likelihood: 
    # How many sensor locations perfectly match the True state vs Particle state?
    def compute_local_likelihood(particle, true_state, x_bounds, y_bounds):
        score = 0
        for sx, sy in zip(sensors_x, sensors_y):
            if x_bounds[0] <= sx < x_bounds[1] and y_bounds[0] <= sy < y_bounds[1]:
                if particle[sy, sx] == true_state[sy, sx]:
                    score += 10 # High reward for match
                else:
                    score += 1  # Low reward for mismatch
        return score

    # --- 4. Filtering Logic ---
    result_fire = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    
    if filter_mode == 'Standard Particle Filter (Global Weight)':
        # Evaluate across the whole map (0 to GRID_SIZE)
        weights =[]
        for p in particles:
            w = compute_local_likelihood(p, true_fire, (0, GRID_SIZE), (0, GRID_SIZE))
            weights.append(w)
            
        # The particle with the highest global weight wins
        best_idx = np.argmax(weights)
        result_fire = particles[best_idx].copy()
        title_str = f"Standard PF Result\n(Chose Particle {best_idx + 1} entirely)"
        
    else: # Spatial Partition PF (2x2 Grid)
        title_str = "Spatial Partition PF Result\n(Stitched Best Local Quadrants)"
        
        # Divide into 4 quadrants
        quadrants =[
            ((0, 20), (0, 20)),   # Top Left (NW)
            ((20, 40), (0, 20)),  # Top Right (NE)
            ((0, 20), (20, 40)),  # Bottom Left (SW)
            ((20, 40), (20, 40))  # Bottom Right (SE)
        ]
        
        for x_bounds, y_bounds in quadrants:
            # Score particles LOCALLY
            local_weights =[]
            for p in particles:
                w = compute_local_likelihood(p, true_fire, x_bounds, y_bounds)
                local_weights.append(w)
            
            # Find the best particle for THIS quadrant
            best_local_idx = np.argmax(local_weights)
            
            # Stitch the winning quadrant into the result
            result_fire[y_bounds[0]:y_bounds[1], x_bounds[0]:x_bounds[1]] = \
                particles[best_local_idx][y_bounds[0]:y_bounds[1], x_bounds[0]:x_bounds[1]]
    
    # --- 5. Plotting ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: True State
    ax1.imshow(true_fire, cmap=cmap, origin='upper')
    ax1.scatter(sensors_x, sensors_y, c='blue', s=40, edgecolor='black', label='Sensors')
    ax1.set_title("True Wildfire State & Sensor Grid", fontweight='bold', fontsize=14)
    ax1.legend(loc='upper right')
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # Draw partition lines if spatial
    if 'Spatial' in filter_mode:
        ax1.axhline(20, color='black', linestyle='--', alpha=0.5)
        ax1.axvline(20, color='black', linestyle='--', alpha=0.5)

    # Plot 2: Filtered State
    ax2.imshow(result_fire, cmap=cmap, origin='upper')
    ax2.set_title(title_str, fontweight='bold', fontsize=14)
    ax2.set_xticks([])
    ax2.set_yticks([])
    
    if 'Spatial' in filter_mode:
        ax2.axhline(20, color='black', linestyle='--', alpha=0.5)
        ax2.axvline(20, color='black', linestyle='--', alpha=0.5)
        
        # Add text to show where the stitched pieces came from
        ax2.text(10, 10, "From P1", color='black', fontweight='bold', ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7))
        ax2.text(30, 30, "From P2", color='black', fontweight='bold', ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7))
        
    # Add borders
    for ax in [ax1, ax2]:
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(2)

    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_spatial_partition, 
         filter_mode=Dropdown(options=['Standard Particle Filter (Global Weight)', 'Spatial Partition PF (2x2 Grid)'], 
                              value='Standard Particle Filter (Global Weight)', 
                              description='Filter Mode:', layout={'width':'500px'}));